In [ ]:
import numpy as np, pandas as pd
from statsmodels.regression.rolling import RollingOLS

PANEL_PATH = "/Users/andradaduta/Desktop/Master Thesis/Python Data/stoxx_600_returns_ff5_mom_usd_2010_2025.parquet"
OUT_PATH   = "stoxx600_AR_2013_2025_w48.parquet"
ID, DATE   = "isin", "date"
Y_COL      = "excess_return_usd"
FACTORS6   = ["Mkt-RF","SMB","HML","RMW","CMA","Mom"]
WINDOW     = 1008          # 48 months; rolling beta window (trading days)
MIN_NOBS   = 1008          # require full window; relax to 806 (~80%) to recover sporadic firms
ANALYSIS_START = "2013-01-01"

panel = pd.read_parquet(PANEL_PATH)
panel[DATE] = pd.to_datetime(panel[DATE])
panel = panel.sort_values([ID, DATE]).reset_index(drop=True)
n0 = len(panel)
panel = panel.dropna(subset=[Y_COL] + FACTORS6).reset_index(drop=True)
print(f"loaded {len(panel):,} rows ({n0-len(panel):,} dropped NaN) | "
      f"{panel[DATE].dt.year.min()}-{panel[DATE].dt.year.max()} | {panel[ID].nunique()} firms")

def add_rolling_ar(df, window=WINDOW, min_nobs=MIN_NOBS):
    cols = ["const"] + FACTORS6
    out = []
    for firm, g in df.groupby(ID, sort=False):
        g = g.sort_values(DATE).copy()
        if len(g) >= min_nobs:
            X = g[FACTORS6].assign(const=1.0)[cols]
            fit = RollingOLS(g[Y_COL], X, window=window, min_nobs=min_nobs).fit()
            betas = fit.params.shift(1)                      # lag: day-t pred uses betas through t-1
            g["pred_excess"] = (betas.values * X.values).sum(axis=1)
            g["beta_mkt"]    = betas["Mkt-RF"].values
        else:
            g["pred_excess"] = np.nan
            g["beta_mkt"]    = np.nan
        g["AR"] = g[Y_COL] - g["pred_excess"]
        out.append(g)
    return pd.concat(out, ignore_index=True)

full = add_rolling_ar(panel)
ar = full[full[DATE] >= ANALYSIS_START].copy()              # trim AFTER computing AR

v = ar.dropna(subset=["AR"])
print(f"\nAR rows (non-NaN): {len(v):,} of {len(ar):,}")
print(f"AR starts        : {v[DATE].min().date()}")
print(f"AR mean / std    : {v['AR'].mean():+.2e} / {v['AR'].std():.4f}")
print(f"rows pre-2013    : {(ar[DATE] < ANALYSIS_START).sum()}  (must be 0)")
print(f"median beta_mkt  : {v['beta_mkt'].median():.2f}")
ar.to_parquet(OUT_PATH)
print(f"saved -> {OUT_PATH}")

In [ ]:
import numpy as np, pandas as pd
import pyfixest as pf

AR_PATH     = "/Users/andradaduta/Desktop/Master Thesis/Python Data/AR/stoxx600_AR_2013_2025_w48.parquet"
EVENTS_PATH = "/Users/andradaduta/Desktop/Master Thesis/Python Data/Events/EU_MSR_Events_sign.xlsx"  # <- your path
ID, DATE = "isin", "date"

INTRO_CLEAN   = [4, 16]              # supply silent (cleanest)
INTRO_NO_ETS2 = [1, 4, 11, 14, 16]  # + reinforcing; drop ETS2-confounded 27  (recommended)
INTRO_ALL     = [1, 4, 11, 14, 16, 27]

ANN_OFFSETS = [-1, 0, 1]
PRE_OFFSETS = [-5, -4, -3, -2]

ar = pd.read_parquet(AR_PATH)
ar[DATE] = pd.to_datetime(ar[DATE])
ar = ar.dropna(subset=["AR"]).sort_values([ID, DATE]).reset_index(drop=True)

# ---- carbon intensity z (FLAG: the AR parquet has no carbon; merge it here) ----
# carbon_intensity_z = log(Scope1/Revenue), within-year z-scored, lagged 1 year, keyed (isin, year).
# If it's already a column on `ar`, delete this block.
carbon = pd.read_parquet("/Users/andradaduta/Desktop/Master Thesis/Python Data/emissions/emissions_revenue_with_isin_key_final.xlsx")    # cols: isin, year, carbon_intensity_z
# ar["year"] = ar[DATE].dt.year
# ar = ar.merge(carbon, on=["isin", "year"], how="left")
assert "carbon_intensity_z" in ar.columns, "Merge carbon_intensity_z first (see block above)."
print(f"carbon merged on {ar['carbon_intensity_z'].notna().mean():.1%} of AR rows")  # sanity

grid = pd.DatetimeIndex(sorted(ar[DATE].unique()))           # trading-day grid = AR panel dates

ev = pd.read_excel(EVENTS_PATH, sheet_name="events")[["event_id","event_date"]]
cs = pd.read_excel(EVENTS_PATH, sheet_name="channel_signs")[["event_id","pv_ar_sign"]]
events = ev.merge(cs, on="event_id")
events["event_date"] = pd.to_datetime(events["event_date"])
events["pv_ar_sign"] = events["pv_ar_sign"].fillna(0).astype(int)

def window_dates(event_date, offsets):
    i = grid.searchsorted(event_date)                        # snap non-trading dates to next trading day
    idx = [i + o for o in offsets if 0 <= i + o < len(grid)]
    return grid[idx]

def build_sx(event_ids, offsets):
    sub = events[events["event_id"].isin(event_ids)]
    date_sign = {}                                           # windows overlap-free -> one sign per date
    for _, r in sub.iterrows():
        for d in window_dates(r["event_date"], offsets):
            date_sign[d] = r["pv_ar_sign"]
    return ar[DATE].map(date_sign).fillna(0.0) * ar["carbon_intensity_z"]   # signed_x_carbon

subsets = {"intro_clean": INTRO_CLEAN, "intro_no_ets2": INTRO_NO_ETS2, "intro_all": INTRO_ALL}
windows = {"announcement [-1,+1]": ANN_OFFSETS, "pre-event [-5,-2]": PRE_OFFSETS}

for wname, offs in windows.items():
    print(f"\n=== {wname} ===")
    for sname, ids in subsets.items():
        ar["sx"] = build_sx(ids, offs)
        res = pf.feols("AR ~ sx | isin + date", data=ar, vcov={"CRV1": "isin + date"})
        r = res.tidy().loc["sx"]
        print(f"  {sname:14s} ({len(ids)} ev): "
              f"beta={r['Estimate']*1000:+.2f}  se={r['Std. Error']*1000:.2f}  "
              f"t={r['t value']:+.2f}  p={r['Pr(>|t|)']:.3f}  "
              f"95%CI=[{r['2.5%']*1000:+.2f}, {r['97.5%']*1000:+.2f}]  "
              f"treated_obs={int((ar['sx']!=0).sum()):,}")